# Python Return Values

> 📘 **Python Mastery** · Module 03 — Functions · Lesson 3/5

Calling a function is like asking a question — `return` is the answer coming back. Master returning one value, many values at once, and returning early, and you will sidestep the single most common beginner bug of them all: printing instead of returning.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Return** a single value and capture it in the caller
- **Return multiple values** and unpack them correctly
- **Write** early returns and guard clauses that keep logic flat
- **Diagnose** the print-vs-return bug that silently produces `None`
- **Build** conditional returns where every path ends in a value
- **Return** boolean expressions directly — and (teaser) whole functions

## 1. Returning One Value

`return` sends a value back to whoever called the function — and it ends the function on the spot. Any code after a `return` that has fired is dead code: it never runs.

The returned value is an ordinary value: store it in a variable, use it inside another expression, or hand it straight to `print`.

**Syntax:**

```python
def function(params):
    ...
    return value        # exits immediately; value goes to the caller
```

**Example:**

In [1]:
def taka_to_usd(taka, rate=117.5):
    """Convert taka to US dollars at a fixed rate."""
    usd = taka / rate
    return usd

    print("Unreachable -- return already exited the function!")   # dead code


wallet = taka_to_usd(5000)          # capture the answer in a variable
print(round(wallet, 2))

# Or use it directly in an expression:
print(taka_to_usd(1000) > 10)

# Returned values chain beautifully: feed one call into the next
def add_service_charge(amount):
    return amount + amount * 0.10


final_bill = add_service_charge(add_service_charge(2000))
print(f"Final bill: {final_bill:.0f} taka")

42.55
False
Final bill: 2420 taka


## 2. Returning Multiple Values

Need a minimum *and* a maximum? A name *and* an age? Just separate the values with commas: `return low, high`. Python packs them automatically, and callers usually unpack the result into variables in one line.

> 🔍 **Under the Hood:** there is no special "multiple return" machinery. The comma builds **one tuple object**, and the function returns that single tuple. Unpacking at the call site is just sequence assignment — which is why mismatched counts raise `ValueError`.

**Syntax:**

```python
def function():
    return value_a, value_b       # really returns ONE tuple

low, high = function()            # tuple unpacking at the call site
```

**Example:**

In [2]:
def min_max(scores):
    """Return the lowest and highest scores as a tuple."""
    return min(scores), max(scores)


lowest, highest = min_max([78, 92, 55, 81])     # unpack into two names
print(lowest, highest)

both = min_max([78, 92, 55, 81])                # or keep the tuple whole
print(both, type(both).__name__)


def stats(numbers):
    """Return three values at once."""
    return min(numbers), max(numbers), sum(numbers) / len(numbers)


lo, hi, avg = stats([78, 92, 55, 81])
print(lo, hi, round(avg, 2))

# Unpacking mismatch -> ValueError
try:
    a, b, c = min_max([1, 2, 3])    # three targets, only two values
except ValueError as err:
    print("ValueError:", err)

55 92
(55, 92) tuple
55 92 76.5
ValueError: not enough values to unpack (expected 3, got 2)


## 3. Early Returns and Guard Clauses

Instead of wrapping your entire function in a giant `if`, handle invalid situations **first** and exit immediately. These quick exits are called **guard clauses**. The main logic then sits flat at the top indentation level — much easier to read than a nested pyramid.

Fail fast, then focus on the happy path.

**Syntax:**

```python
def function(value):
    if bad_input(value):
        return fallback              # guard clause: leave early
    return do_the_real_work(value)   # happy path stays unindented
```

**Example:**

In [3]:
def discount_for(age, is_student):
    """Return the discount rate, guarding against odd input first."""
    if age < 0:
        return 0.0               # guard: impossible age, stop here
    if age < 5:
        return 0.5               # tiny children travel nearly free
    if is_student:
        return 0.2
    if age >= 60:
        return 0.3
    return 0.0                   # everyone else


for age, student in [(4, False), (21, True), (-3, True), (65, False)]:
    print(f"age={age}, student={student} -> {discount_for(age, student):.0%}")

age=4, student=False -> 50%
age=21, student=True -> 20%
age=-3, student=True -> 0%
age=65, student=False -> 30%


In [4]:
# Side by side: the nested pyramid vs the flat guard-clause version
def withdraw_pyramid(balance, amount):
    if amount > 0:
        if balance >= amount:
            balance -= amount
            return balance
        else:
            return None              # insufficient funds
    else:
        return None                  # invalid amount


def withdraw_flat(balance, amount):
    if amount <= 0:
        return None                  # guard 1
    if balance < amount:
        return None                  # guard 2
    return balance - amount          # happy path


print(withdraw_pyramid(5000, 1200), withdraw_flat(5000, 1200))
print(withdraw_pyramid(5000, -5), withdraw_flat(5000, -5))

3800 3800
None None


In [5]:
# `return` also breaks out of a loop the moment you find what you want
def first_even(numbers):
    for n in numbers:
        if n % 2 == 0:
            return n                 # early exit: no need to keep scanning
    return None                      # loop finished: nothing even found


print(first_even([7, 3, 8, 10]))
print(first_even([1, 3, 5]))

8
None


## 4. ⚠️ Forgetting `return`: the Silent `None`

The classic beginner bug. This function *looks* finished:

```python
def add_tax(price):
    print(price * 1.15)      # shows the answer... but returns None
```

`print` puts text **on the screen**; `return` hands a value **back to the program**. They are completely different actions. A function with no `return` (or a bare `return`) delivers `None` — so `total = add_tax(100)` stores `None`, and the crash appears later, far from the real mistake.

❌

```python
def average_bad(scores):
    print(sum(scores) / len(scores))     # caller receives None
```

✅

```python
def average_good(scores):
    return sum(scores) / len(scores)     # caller decides what to do with it
```

> 🔍 **Under the Hood:** when execution falls off the end of a function body, CPython implicitly executes `return None`. Every function call is an expression that evaluates to exactly one object — if you did not choose it, Python chose `None` for you.

**Example:** watch `None` sneak into a variable.

In [6]:
def average_prints(scores):
    print("Inside:", sum(scores) / len(scores))     # screen only


def average_returns(scores):
    return sum(scores) / len(scores)                # usable value


result_a = average_prints([70, 80, 90])
print("Captured from print-version:", result_a)     # None!

result_b = average_returns([70, 80, 90])
print("Captured from return-version:", result_b)    # 80.0

# The delayed crash: None cannot take part in arithmetic
try:
    bonus = average_prints([70, 80, 90]) * 1.05
except TypeError as err:
    print("TypeError:", err)

Inside: 80.0
Captured from print-version: None
Captured from return-version: 80.0
Inside: 80.0
TypeError: unsupported operand type(s) for *: 'NoneType' and 'float'


In [7]:
# Three ways to "not return" -- all produce exactly None
def silent():           # no return statement at all
    x = 1

def bare_return():      # return with nothing after it
    x = 1
    return

def halfway(flag):      # only SOME paths return a value
    if flag:
        return 42


print(silent(), bare_return(), halfway(True), halfway(False))

None None 42 None


## 5. Conditional Returns

Real decisions need branches. The golden rule: make sure **every path** through the function ends in a `return`. A branch that falls off the edge silently yields `None` — and mixed return types (`float` sometimes, `None` other times) confuse every caller downstream.

**Syntax:**

```python
def classify(score):
    if score >= high_cut:
        return "A"
    elif score >= low_cut:
        return "B"
    else:
        return "F"        # catch-all: no path escapes
```

**Example:**

In [8]:
def grade_letter(score):
    """Map an exam score (out of 100) to a letter grade."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= 60:
        return "D"
    else:
        return "F"


for s in (95, 84, 71, 63, 40):
    print(f"{s} -> {grade_letter(s)}")

# A missing else = accidental None on some inputs
def risky_grade(score):
    if score >= 33:
        return "Pass"      # scores below 33 fall off the end...


print(risky_grade(90))
print(risky_grade(20))     # ...and None sneaks out

95 -> A
84 -> B
71 -> C
63 -> D
40 -> F
Pass
None


## 6. Returning Boolean Expressions Directly

Beginners often write:

```python
if condition:
    return True
else:
    return False
```

But the condition **already is** a boolean — so return it directly. Fewer lines, identical behaviour, and it reads like English. This style also composes nicely with `and` / `or` for compound questions.

**Syntax:**

```python
def check(value):
    return value > limit        # the comparison IS the True/False
```

**Example:**

In [9]:
def is_adult(age):
    return age >= 18                    # no if needed!


def has_passing_grade(marks):
    return marks >= 33


print(is_adult(21), is_adult(15))
print(has_passing_grade(90), has_passing_grade(12))


# Combine conditions for richer predicates
def gets_concession(age, is_student):
    return age < 18 or is_student


print(gets_concession(16, False), gets_concession(25, True))

# Boolean-returning functions plug straight into filters
ages = [12, 19, 34, 17, 65]
adults_only = [a for a in ages if is_adult(a)]
print(adults_only)

True False
True False
True True
[19, 34, 65]


## 7. Returning a Function (Teaser)

Since functions are objects (Lesson 1), a function can **build and return another function**. Here `make_multiplier` returns a customised machine: the inner function remembers the `factor` it was born with. How it remembers is the magic of *closures* — full story in Lesson 4. For now, enjoy factories that stamp out behaviour on demand.

**Syntax:**

```python
def make_x(config):
    def inner(value):
        ...use config...
    return inner          # hand back the FUNCTION itself -- no parentheses!

doubler = make_x(2)       # doubler is now a function
doubler(10)               # -> 20
```

**Example:**

In [10]:
def make_multiplier(factor):
    """Return a brand-new function that multiplies by factor."""
    def multiply(number):
        return number * factor
    return multiply            # return the FUNCTION -- do not call it here


double = make_multiplier(2)
triple = make_multiplier(3)

print(double(10), triple(10))
print(make_multiplier(5)(100))         # build and call in one breath
print(type(double).__name__)           # just an ordinary function object

20 30
500
function


In [11]:
# One factory, many behaviours
def make_power(n):
    def power(x):
        return x ** n
    return power


square = make_power(2)
cube = make_power(3)
sqrt = make_power(0.5)

print(square(9), cube(3), sqrt(49))

# Each returned function kept ITS OWN n -- next lesson explains how.

81 27 7.0


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
| --- | --- | --- |
| `print()` where a result is needed | Caller receives `None`; the crash surfaces later, far from the cause | `return` the value; print only at the edges of your program |
| Code placed after `return` | Unreachable — never executed | Delete it, or move it above the `return` |
| Some branches return, others fall off the end | Silent `None` for missed cases | Give every branch a `return`; finish with a catch-all `else`/final `return` |
| Inconsistent return types (list today, `None` tomorrow, string later) | Every caller needs special-case checks | Pick one return type; signal bad input with an exception instead |
| Unpacking the wrong number of values | `ValueError: too many values to unpack` | Match variable count to what the function actually returns |

## 💡 Best Practices & Pro Tips

- **Return data, print at the boundaries.** Functions compute; the main script (or UI) decides what to display.
- Put guard clauses first so the happy path reads top-to-bottom without nesting.
- Name fruitful functions after what they give back: `get_balance`, `calculate_fare`, `is_valid`.
- Prefer **pure functions** — same inputs always produce the same returned output, no hidden side effects. They are trivially testable.
- 🤖 **AI-engineering relevance:** scikit-learn's API is built on this discipline — `.fit()` and `.predict()` *return* fitted estimators and arrays rather than printing summaries, which is why models compose into pipelines. Factory functions that return closures power custom transformers and image/text augmentations in PyTorch.

## 📌 Summary

| Tool | What it does | Example |
| --- | --- | --- |
| `return v` | Sends one value back, ends the call | `return w * h` |
| `return a, b` | Packs values into a tuple | `return lo, hi` |
| `x, y = f()` | Tuple unpacking at the call site | `low, high = min_max(s)` |
| *(no return)* | Implicit `return None` | `print(f())` → `None` |
| `return` alone / mid-function | Exits early with `None` (or a chosen value) | guard clauses |
| `return condition` | Boolean expression returned directly | `return age >= 18` |
| `def outer(): ... return inner` | Function factory (closure teaser) | `double = make_multiplier(2)` |

Key takeaways:

- `print` entertains humans; `return` feeds programs. Confusing them is bug #1.
- Multiple values are one tuple — unpack them or lose the structure.
- Guard early, return often: every path should deliver a value.
- Functions can return functions — the doorway to closures and decorators.

## 🔗 Next Lesson

➡️ Continue with **04_Scope** — where variables live, the LEGB lookup rule, why `count += 1` inside a function explodes, and the closures teased today.